# Phase 3 v2.0: Feature Selection & Correlation Analysis

## Objective

Optimize the feature set from Phase 2B Enhanced by:
1. **Correlation Analysis** - Identify and remove highly correlated features (>0.9)
2. **Feature Variance Analysis** - Remove low-variance features
3. **Feature Importance Analysis** - Identify most predictive features
4. **Missing Data Analysis** - Handle features with excessive missing values
5. **Final Feature Set Selection** - Create optimized dataset for model training

**Input**: Phase 2B Enhanced output (283,118 records, 62 columns)  
**Output**: Optimized feature set for model training

---

## Important Constraints

**MUST PRESERVE**: 
- `lf_target_vcn` and `lf_cluster_index` - Required by Oh et al. (2024) Algorithm 2
- `timestomped` - Target variable
- All identifier columns - `case_id`, `eventtime`, `filename`, etc.

**CAN REMOVE**:
- Highly correlated features (>0.9 correlation)
- Low variance features
- Features with excessive missing data (>95%)

---

## 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Define paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
INPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 2B - V2 Column Cleanup'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 3 - V2 Feature Selection'

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Directory Configuration:")
print(f"  Input: {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Output exists: {OUTPUT_DIR.exists()}")

Directory Configuration:
  Input: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2B - V2 Column Cleanup
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection
  Output exists: True


In [3]:
# Load Phase 2B Enhanced output
print("Loading Phase 2B Enhanced dataset...")
input_file = INPUT_DIR / 'all_cases_combined_v2_phase2b_enhanced.csv'

df = pd.read_csv(input_file, encoding='utf-8-sig')

print(f"\nDataset loaded successfully:")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Timestomped events: {(df['timestomped'] == 1).sum():,}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Loading Phase 2B Enhanced dataset...

Dataset loaded successfully:
  Records: 283,118
  Columns: 62
  Timestomped events: 280
  Memory usage: 437.55 MB


---
## 2. Data Overview & Column Categorization

In [4]:
print("=" * 80)
print("COLUMN OVERVIEW")
print("=" * 80)

print(f"\nAll 62 columns with data types and coverage:")
for i, col in enumerate(df.columns, 1):
    non_null = df[col].notna().sum()
    pct = (non_null / len(df)) * 100
    dtype = df[col].dtype
    print(f"{i:2d}. {col:45s} - {str(dtype):10s} - {non_null:7,} non-null ({pct:5.1f}%)")

COLUMN OVERVIEW

All 62 columns with data types and coverage:
 1. case_id                                       - object     - 283,118 non-null (100.0%)
 2. eventtime                                     - object     - 282,702 non-null ( 99.9%)
 3. eventtime_dt                                  - object     - 282,702 non-null ( 99.9%)
 4. lf_lsn                                        - float64    -   4,725 non-null (  1.7%)
 5. lf_event                                      - object     -   4,725 non-null (  1.7%)
 6. filename                                      - object     - 283,118 non-null (100.0%)
 7. filepath                                      - object     - 217,542 non-null ( 76.8%)
 8. lf_target_vcn                                 - object     -   4,725 non-null (  1.7%)
 9. lf_cluster_index                              - float64    -   4,725 non-null (  1.7%)
10. merge_key                                     - object     - 283,118 non-null (100.0%)
11. usn_usn                 

In [5]:
# Categorize columns by their purpose
print("\n" + "=" * 80)
print("COLUMN CATEGORIZATION")
print("=" * 80)

# Identifier columns (not used for modeling)
identifier_cols = [
    'case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key'
]

# Target variable
target_col = 'timestomped'

# Forensic metadata (low coverage but forensically critical)
forensic_metadata_cols = [
    'lf_lsn', 'lf_event', 'lf_target_vcn', 'lf_cluster_index',
    'usn_usn', 'usn_event_info', 'usn_file_reference_number',
    'usn_parent_file_reference_number'
]

# Source information
source_cols = ['source']

# Parsed timestamps (low coverage)
parsed_timestamp_cols = [
    'lf_creation_time_before', 'lf_creation_time_after',
    'lf_modified_time_before', 'lf_modified_time_after',
    'lf_accessed_time_before', 'lf_accessed_time_after',
    'lf_mft_modified_time_before', 'lf_mft_modified_time_after'
]

# Timestamp delta features (derived from parsed timestamps)
timestamp_delta_cols = [
    'creation_time_delta_days', 'creation_time_changed_to_past',
    'modified_time_delta_days', 'modified_time_changed_to_past',
    'accessed_time_delta_days', 'accessed_time_changed_to_past',
    'mft_modified_time_delta_days', 'mft_modified_time_changed_to_past'
]

# Binary forensic indicators
forensic_indicator_cols = [
    'zero_in_nanoseconds', 'copied_from_file', 'is_tunneling',
    'has_attribute_change', 'has_timestamp_copied_from_file', 'zero_nanoseconds_logfile'
]

# File attributes
file_attribute_cols = [
    'is_executable', 'is_system_file', 'is_hidden_file', 'is_archive',
    'filename_length', 'has_suspicious_extension', 'path_depth',
    'usn_file_attribute'
]

# Temporal features
temporal_cols = [
    'event_frequency_per_file', 'event_frequency_per_case',
    'events_in_1min_window', 'events_in_5min_window',
    'time_since_previous_event_seconds', 'time_until_next_event_seconds',
    'event_vs_modified_after_days'
]

# Confidence/validation scores
score_cols = [
    'source_confidence_score', 'cross_artifact_validation_score',
    'timestamp_manipulation_pattern_score', 'file_system_tunneling_confidence'
]

# Evidence flags
evidence_cols = [
    'has_logfile_evidence', 'has_usnjrnl_evidence'
]

# USN Journal patterns
usn_pattern_cols = [
    'usn_basic_info_change', 'usn_file_closed', 'usn_complete_manipulation_pattern'
]

print(f"\nIdentifier columns: {len(identifier_cols)}")
print(f"Target variable: 1")
print(f"Forensic metadata: {len(forensic_metadata_cols)}")
print(f"Source: {len(source_cols)}")
print(f"Parsed timestamps: {len(parsed_timestamp_cols)}")
print(f"Timestamp deltas: {len(timestamp_delta_cols)}")
print(f"Forensic indicators: {len(forensic_indicator_cols)}")
print(f"File attributes: {len(file_attribute_cols)}")
print(f"Temporal features: {len(temporal_cols)}")
print(f"Scores: {len(score_cols)}")
print(f"Evidence flags: {len(evidence_cols)}")
print(f"USN patterns: {len(usn_pattern_cols)}")
print(f"\nTotal: {len(identifier_cols) + 1 + len(forensic_metadata_cols) + len(source_cols) + len(parsed_timestamp_cols) + len(timestamp_delta_cols) + len(forensic_indicator_cols) + len(file_attribute_cols) + len(temporal_cols) + len(score_cols) + len(evidence_cols) + len(usn_pattern_cols)}")


COLUMN CATEGORIZATION

Identifier columns: 6
Target variable: 1
Forensic metadata: 8
Source: 1
Parsed timestamps: 8
Timestamp deltas: 8
Forensic indicators: 6
File attributes: 8
Temporal features: 7
Scores: 4
Evidence flags: 2
USN patterns: 3

Total: 62


---
## 3. Missing Data Analysis

In [6]:
print("=" * 80)
print("MISSING DATA ANALYSIS")
print("=" * 80)

# Calculate missing percentages
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

print(f"\nColumns with missing data:")
missing_cols = missing_pct[missing_pct > 0]
for col, pct in missing_cols.items():
    print(f"  {col:45s}: {pct:5.1f}% missing")

# Identify columns with >95% missing data
high_missing = missing_pct[missing_pct > 95]
print(f"\n" + "=" * 80)
print(f"Columns with >95% missing data: {len(high_missing)}")
print("=" * 80)
if len(high_missing) > 0:
    for col, pct in high_missing.items():
        print(f"  {col:45s}: {pct:5.1f}% missing")
else:
    print("  None")

MISSING DATA ANALYSIS

Columns with missing data:
  accessed_time_delta_days                     :  99.9% missing
  creation_time_delta_days                     :  99.8% missing
  lf_accessed_time_before                      :  99.8% missing
  lf_accessed_time_after                       :  99.8% missing
  lf_creation_time_after                       :  99.5% missing
  lf_creation_time_before                      :  99.5% missing
  mft_modified_time_delta_days                 :  99.5% missing
  lf_mft_modified_time_before                  :  99.3% missing
  lf_mft_modified_time_after                   :  99.3% missing
  event_vs_modified_after_days                 :  99.0% missing
  modified_time_delta_days                     :  99.0% missing
  lf_modified_time_after                       :  98.7% missing
  lf_modified_time_before                      :  98.7% missing
  lf_cluster_index                             :  98.3% missing
  lf_target_vcn                                :  98.3

---
## 4. Identify Feature Columns for Analysis

Separate columns into:
- **Identifiers**: Not used for modeling (case_id, filename, etc.)
- **Target**: timestomped
- **Features**: All potential modeling features

In [7]:
print("=" * 80)
print("IDENTIFYING FEATURE COLUMNS")
print("=" * 80)

# Columns to exclude from feature analysis
non_feature_cols = identifier_cols + [target_col]

# All feature columns
feature_cols = [col for col in df.columns if col not in non_feature_cols]

print(f"\nTotal columns: {len(df.columns)}")
print(f"Non-feature columns: {len(non_feature_cols)}")
print(f"  - Identifiers: {len(identifier_cols)}")
print(f"  - Target: 1")
print(f"\nFeature columns for analysis: {len(feature_cols)}")

print(f"\nFeature columns:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

IDENTIFYING FEATURE COLUMNS

Total columns: 62
Non-feature columns: 7
  - Identifiers: 6
  - Target: 1

Feature columns for analysis: 55

Feature columns:
   1. lf_lsn
   2. lf_event
   3. lf_target_vcn
   4. lf_cluster_index
   5. usn_usn
   6. usn_event_info
   7. usn_file_attribute
   8. usn_file_reference_number
   9. usn_parent_file_reference_number
  10. source
  11. is_tunneling
  12. lf_creation_time_before
  13. lf_creation_time_after
  14. lf_modified_time_before
  15. lf_modified_time_after
  16. lf_accessed_time_before
  17. lf_accessed_time_after
  18. lf_mft_modified_time_before
  19. lf_mft_modified_time_after
  20. zero_in_nanoseconds
  21. copied_from_file
  22. creation_time_delta_days
  23. creation_time_changed_to_past
  24. modified_time_delta_days
  25. modified_time_changed_to_past
  26. accessed_time_delta_days
  27. accessed_time_changed_to_past
  28. mft_modified_time_delta_days
  29. mft_modified_time_changed_to_past
  30. is_executable
  31. is_system_file
 

---
## 5. Select Numeric Features for Correlation Analysis

We'll analyze correlations only on numeric features that are suitable for ML models.

In [8]:
print("=" * 80)
print("SELECTING NUMERIC FEATURES")
print("=" * 80)

# Select only numeric columns from features
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# Separate numeric features by type
int_features = df[feature_cols].select_dtypes(include=['int64', 'int32']).columns.tolist()
float_features = df[feature_cols].select_dtypes(include=['float64', 'float32']).columns.tolist()

print(f"\nNumeric features: {len(numeric_features)}")
print(f"  - Integer features: {len(int_features)}")
print(f"  - Float features: {len(float_features)}")

# Non-numeric features (strings, objects)
non_numeric_features = [col for col in feature_cols if col not in numeric_features]
print(f"  - Non-numeric features: {len(non_numeric_features)}")

if len(non_numeric_features) > 0:
    print(f"\nNon-numeric features to exclude from correlation analysis:")
    for col in non_numeric_features:
        print(f"  - {col}: {df[col].dtype}")

SELECTING NUMERIC FEATURES

Numeric features: 23
  - Integer features: 12
  - Float features: 11
  - Non-numeric features: 32

Non-numeric features to exclude from correlation analysis:
  - lf_event: object
  - lf_target_vcn: object
  - usn_event_info: object
  - usn_file_attribute: object
  - usn_file_reference_number: object
  - usn_parent_file_reference_number: object
  - source: object
  - is_tunneling: bool
  - lf_creation_time_before: object
  - lf_creation_time_after: object
  - lf_modified_time_before: object
  - lf_modified_time_after: object
  - lf_accessed_time_before: object
  - lf_accessed_time_after: object
  - lf_mft_modified_time_before: object
  - lf_mft_modified_time_after: object
  - zero_in_nanoseconds: bool
  - copied_from_file: object
  - creation_time_changed_to_past: object
  - modified_time_changed_to_past: object
  - accessed_time_changed_to_past: object
  - mft_modified_time_changed_to_past: object
  - is_executable: bool
  - is_system_file: bool
  - is_hidde

---
## 6. Correlation Analysis

Calculate correlation matrix for numeric features and identify highly correlated pairs.

In [9]:
print("=" * 80)
print("CALCULATING CORRELATION MATRIX")
print("=" * 80)

# Calculate correlation matrix for numeric features
print(f"\nCalculating correlations for {len(numeric_features)} numeric features...")
print("This may take a minute...")

corr_matrix = df[numeric_features].corr()

print(f"\nCorrelation matrix calculated: {corr_matrix.shape[0]} x {corr_matrix.shape[1]}")
print(f"Memory usage: {corr_matrix.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

CALCULATING CORRELATION MATRIX

Calculating correlations for 23 numeric features...
This may take a minute...

Correlation matrix calculated: 23 x 23
Memory usage: 0.01 MB


In [10]:
# Find highly correlated feature pairs (|correlation| > 0.9)
print("\n" + "=" * 80)
print("HIGHLY CORRELATED FEATURES (|correlation| > 0.9)")
print("=" * 80)

# Get upper triangle of correlation matrix
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find pairs with |correlation| > 0.9
high_corr_pairs = []
for column in upper_triangle.columns:
    for index in upper_triangle.index:
        corr_value = upper_triangle.loc[index, column]
        if pd.notna(corr_value) and abs(corr_value) > 0.9:
            high_corr_pairs.append({
                'Feature 1': index,
                'Feature 2': column,
                'Correlation': corr_value
            })

if len(high_corr_pairs) > 0:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print(f"\nFound {len(high_corr_pairs)} highly correlated pairs:")
    print(high_corr_df.to_string(index=False))
else:
    print("\nNo highly correlated pairs found (|correlation| > 0.9)")
    high_corr_df = pd.DataFrame()


HIGHLY CORRELATED FEATURES (|correlation| > 0.9)

Found 8 highly correlated pairs:
                        Feature 1                       Feature 2  Correlation
         modified_time_delta_days    event_vs_modified_after_days    -1.000000
         accessed_time_delta_days    mft_modified_time_delta_days     1.000000
         creation_time_delta_days        accessed_time_delta_days     1.000000
         creation_time_delta_days    mft_modified_time_delta_days     1.000000
                           lf_lsn                         usn_usn     0.985471
          source_confidence_score cross_artifact_validation_score     0.984313
time_since_previous_event_seconds   time_until_next_event_seconds     0.954919
            events_in_1min_window           events_in_5min_window     0.927746


In [11]:
# Correlation with target variable
print("\n" + "=" * 80)
print("CORRELATION WITH TARGET VARIABLE (timestomped)")
print("=" * 80)

# Calculate correlation with target
target_corr = df[numeric_features + [target_col]].corr()[target_col].drop(target_col)
target_corr_sorted = target_corr.abs().sort_values(ascending=False)

print(f"\nTop 20 features most correlated with timestomped:")
for i, (feature, corr) in enumerate(target_corr_sorted.head(20).items(), 1):
    actual_corr = target_corr[feature]
    print(f"  {i:2d}. {feature:45s}: {actual_corr:7.4f} (|{abs(actual_corr):.4f}|)")

print(f"\nBottom 10 features least correlated with timestomped:")
for i, (feature, corr) in enumerate(target_corr_sorted.tail(10).items(), 1):
    actual_corr = target_corr[feature]
    print(f"  {i:2d}. {feature:45s}: {actual_corr:7.4f} (|{abs(actual_corr):.4f}|)")


CORRELATION WITH TARGET VARIABLE (timestomped)

Top 20 features most correlated with timestomped:
   1. accessed_time_delta_days                     :  0.6539 (|0.6539|)
   2. creation_time_delta_days                     :  0.1379 (|0.1379|)
   3. mft_modified_time_delta_days                 :  0.0446 (|0.0446|)
   4. event_frequency_per_case                     : -0.0421 (|0.0421|)
   5. modified_time_delta_days                     : -0.0317 (|0.0317|)
   6. timestamp_manipulation_pattern_score         :  0.0297 (|0.0297|)
   7. time_since_previous_event_seconds            :  0.0274 (|0.0274|)
   8. time_until_next_event_seconds                :  0.0274 (|0.0274|)
   9. usn_usn                                      :  0.0273 (|0.0273|)
  10. event_vs_modified_after_days                 :  0.0245 (|0.0245|)
  11. events_in_5min_window                        : -0.0188 (|0.0188|)
  12. cross_artifact_validation_score              :  0.0170 (|0.0170|)
  13. events_in_1min_window          

---
## 7. Variance Analysis

Identify low-variance features that may not be useful for modeling.

In [12]:
print("=" * 80)
print("VARIANCE ANALYSIS")
print("=" * 80)

# Calculate variance for numeric features
variances = df[numeric_features].var().sort_values()

print(f"\nFeatures with lowest variance:")
for i, (feature, var) in enumerate(variances.head(15).items(), 1):
    unique_vals = df[feature].nunique()
    print(f"  {i:2d}. {feature:45s}: variance={var:12.6f}, unique_values={unique_vals}")

# Identify near-zero variance features
zero_var_features = variances[variances < 0.01].index.tolist()
print(f"\nFeatures with near-zero variance (<0.01): {len(zero_var_features)}")
if len(zero_var_features) > 0:
    for feat in zero_var_features:
        print(f"  - {feat}")

VARIANCE ANALYSIS

Features with lowest variance:
   1. has_attribute_change                         : variance=    0.001819, unique_values=2
   2. has_timestamp_copied_from_file               : variance=    0.002888, unique_values=2
   3. zero_nanoseconds_logfile                     : variance=    0.005329, unique_values=2
   4. file_system_tunneling_confidence             : variance=    0.008908, unique_values=2
   5. source_confidence_score                      : variance=    0.014594, unique_values=2
   6. cross_artifact_validation_score              : variance=    0.060138, unique_values=3
   7. timestamp_manipulation_pattern_score         : variance=    0.441075, unique_values=4
   8. lf_cluster_index                             : variance=    4.818337, unique_values=4
   9. path_depth                                   : variance=   10.475849, unique_values=14
  10. filename_length                              : variance=  875.816113, unique_values=123
  11. accessed_time_delta_d

---
## 8. Recommendations for Feature Removal

Based on correlation analysis, missing data, and variance analysis.

In [13]:
print("=" * 80)
print("FEATURE REMOVAL RECOMMENDATIONS")
print("=" * 80)

# Features to consider removing
removal_candidates = []

# 1. High correlation - keep one from each pair
if len(high_corr_df) > 0:
    print(f"\n1. Highly correlated features (|r| > 0.9):")
    for _, row in high_corr_df.iterrows():
        feat1, feat2, corr = row['Feature 1'], row['Feature 2'], row['Correlation']
        # Keep the one with higher correlation to target
        if feat1 in target_corr.index and feat2 in target_corr.index:
            corr1 = abs(target_corr[feat1])
            corr2 = abs(target_corr[feat2])
            to_remove = feat2 if corr1 >= corr2 else feat1
            to_keep = feat1 if corr1 >= corr2 else feat2
            print(f"  - Remove: {to_remove:45s} (keep {to_keep}, r={corr:.4f})")
            removal_candidates.append(to_remove)
else:
    print(f"\n1. No highly correlated features found")

# 2. Near-zero variance
print(f"\n2. Near-zero variance features (<0.01):")
if len(zero_var_features) > 0:
    for feat in zero_var_features:
        print(f"  - {feat}")
        removal_candidates.append(feat)
else:
    print("  None")

# 3. High missing data (>95%)
print(f"\n3. Features with >95% missing data:")
if len(high_missing) > 0:
    for col in high_missing.index:
        if col in feature_cols:  # Only feature columns
            print(f"  - {col:45s} ({high_missing[col]:.1f}% missing)")
            removal_candidates.append(col)
else:
    print("  None")

# Deduplicate removal candidates
removal_candidates = list(set(removal_candidates))

print(f"\n" + "=" * 80)
print(f"Total features recommended for removal: {len(removal_candidates)}")
print("=" * 80)

if len(removal_candidates) > 0:
    for i, feat in enumerate(sorted(removal_candidates), 1):
        print(f"  {i:2d}. {feat}")

FEATURE REMOVAL RECOMMENDATIONS

1. Highly correlated features (|r| > 0.9):
  - Remove: event_vs_modified_after_days                  (keep modified_time_delta_days, r=-1.0000)
  - Remove: mft_modified_time_delta_days                  (keep accessed_time_delta_days, r=1.0000)
  - Remove: creation_time_delta_days                      (keep accessed_time_delta_days, r=1.0000)
  - Remove: mft_modified_time_delta_days                  (keep creation_time_delta_days, r=1.0000)
  - Remove: lf_lsn                                        (keep usn_usn, r=0.9855)
  - Remove: source_confidence_score                       (keep cross_artifact_validation_score, r=0.9843)
  - Remove: time_until_next_event_seconds                 (keep time_since_previous_event_seconds, r=0.9549)
  - Remove: events_in_1min_window                         (keep events_in_5min_window, r=0.9277)

2. Near-zero variance features (<0.01):
  - has_attribute_change
  - has_timestamp_copied_from_file
  - zero_nanoseconds_logfi

---
## 9. Summary Report

In [14]:
print("\n" + "=" * 80)
print("PHASE 3 v2.0 SUMMARY REPORT")
print("=" * 80)

print("\nPHASE 3 v2.0 COMPLETE - FEATURE SELECTION & CORRELATION ANALYSIS")

print("\n" + "=" * 80)
print("1. DATASET OVERVIEW")
print("=" * 80)
print(f"  Total columns: {len(df.columns)}")
print(f"  Identifier columns: {len(identifier_cols)}")
print(f"  Target variable: 1")
print(f"  Feature columns: {len(feature_cols)}")
print(f"    - Numeric: {len(numeric_features)}")
print(f"    - Non-numeric: {len(non_numeric_features)}")

print("\n" + "=" * 80)
print("2. CORRELATION ANALYSIS")
print("=" * 80)
print(f"  Highly correlated pairs (|r| > 0.9): {len(high_corr_df)}")
if len(high_corr_df) > 0:
    print(f"  Strongest correlation: {high_corr_df.iloc[0]['Correlation']:.4f}")

print("\n" + "=" * 80)
print("3. TARGET CORRELATION")
print("=" * 80)
print(f"  Top feature correlation with timestomped:")
top_feature = target_corr_sorted.index[0]
print(f"    {top_feature}: {target_corr[top_feature]:.4f}")

print("\n" + "=" * 80)
print("4. VARIANCE ANALYSIS")
print("=" * 80)
print(f"  Near-zero variance features (<0.01): {len(zero_var_features)}")

print("\n" + "=" * 80)
print("5. MISSING DATA")
print("=" * 80)
print(f"  Features with >95% missing: {len(high_missing)}")
print(f"  Features with any missing: {len(missing_cols)}")

print("\n" + "=" * 80)
print("6. RECOMMENDATIONS")
print("=" * 80)
print(f"  Features recommended for removal: {len(removal_candidates)}")
if len(removal_candidates) > 0:
    print(f"  Reasons:")
    if len(high_corr_df) > 0:
        print(f"    - High correlation: {len([f for f in removal_candidates if f in high_corr_df['Feature 2'].values or f in high_corr_df['Feature 1'].values])}")
    if len(zero_var_features) > 0:
        print(f"    - Near-zero variance: {len([f for f in removal_candidates if f in zero_var_features])}")
    if len(high_missing) > 0:
        print(f"    - High missing data: {len([f for f in removal_candidates if f in high_missing.index])}")

print(f"\n  Projected columns after removal: {len(df.columns) - len(removal_candidates)}")
print(f"  Projected feature columns: {len(feature_cols) - len(removal_candidates)}")

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("\n1. Review removal recommendations")
print("2. Decide which features to remove")
print("3. Create final feature set for model training (Phase 4)")
print("4. Proceed with model training and evaluation")


PHASE 3 v2.0 SUMMARY REPORT

PHASE 3 v2.0 COMPLETE - FEATURE SELECTION & CORRELATION ANALYSIS

1. DATASET OVERVIEW
  Total columns: 62
  Identifier columns: 6
  Target variable: 1
  Feature columns: 55
    - Numeric: 23
    - Non-numeric: 32

2. CORRELATION ANALYSIS
  Highly correlated pairs (|r| > 0.9): 8
  Strongest correlation: -1.0000

3. TARGET CORRELATION
  Top feature correlation with timestomped:
    accessed_time_delta_days: 0.6539

4. VARIANCE ANALYSIS
  Near-zero variance features (<0.01): 4

5. MISSING DATA
  Features with >95% missing: 17
  Features with any missing: 30

6. RECOMMENDATIONS
  Features recommended for removal: 24
  Reasons:
    - High correlation: 9
    - Near-zero variance: 4
    - High missing data: 17

  Projected columns after removal: 38
  Projected feature columns: 31

NEXT STEPS

1. Review removal recommendations
2. Decide which features to remove
3. Create final feature set for model training (Phase 4)
4. Proceed with model training and evaluation
